<a href="https://colab.research.google.com/github/hasmalee/aeropinn/blob/main/AeroPINNN_X_XAI_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AeroPINN_X_v3_Colab.py
# Final stage-runner notebook script
# ==================================

from __future__ import annotations

import os
import gc
import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from airfoil import AirfoilGeometry, AirfoilFormatError
from parsec_fit import (
    fit_parsec_to_dat,
    fit_quality_report,
    preprocess_airfoil_for_parsec,
)
from network import MLP, load_checkpoint, save_checkpoint
from train_loop import (
    train_multi_airfoil,
    optimize_shape,
)
from postprocess import (
    validate_model_on_library,
    export_optimized_airfoil,
    export_run_report,
)
from xai import run_xai_suite

In [ ]:
# ============================================================================
# CELL 1 ─ Environment
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive/MyDrive')

In [ ]:
# ============================================================================
# CELL 2 ─ User stage selection
# ============================================================================
# STAGE = 1 -> pilot
# STAGE = 2 -> research training
# STAGE = 3 -> optimize one selected airfoil using trained model
STAGE = 2

# folders
DATA_ROOT = Path("/content/airfoils")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")
CKPT_DIR = OUT_DIR / "checkpoints"
VAL_OUT_DIR = OUT_DIR / "validation"
OPT_OUT_DIR = OUT_DIR / "optimized_results"
XAI_OUT_DIR = OUT_DIR / "xai_outputs"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
OPT_OUT_DIR.mkdir(parents=True, exist_ok=True)
XAI_OUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_CHECKPOINT = None   # e.g. CKPT_DIR / "multi_airfoil_last.pt"



In [ ]:
# ============================================================================
# CELL 3 ─ Config
# ============================================================================
if STAGE == 1:
    AOA_LIST = [-2, 0, 2, 4]
    RE_LIST = [5e4, 1e5]

    N_INT = 2000
    N_NEAR = 400
    N_WAKE = 400
    N_AIRFOIL = 600
    N_INLET = 400
    N_OUTLET = 400
    N_TOP = 400
    N_BOT = 400

    ADAM_STEPS = 1500
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 300
    ADAPTIVE_CAP = 1000
    CANDIDATE_POOL = 4000
    GAUSS_PER_CENTER = 4
    RESIDUAL_PROBE_STRIDE = 6

elif STAGE == 2:
    AOA_LIST = [0.0, 4.0]
    RE_LIST = [1e6, 2e6]

    N_INT = 3000
    N_NEAR = 600
    N_WAKE = 600
    N_AIRFOIL = 800
    N_INLET = 500
    N_OUTLET = 500
    N_TOP = 500
    N_BOT = 500

    ADAM_STEPS = 5000
    LBFGS_STEPS = 10
    ADAPTIVE_EVERY = 100
    ADAPTIVE_CAP = 2000
    CANDIDATE_POOL = 6000
    GAUSS_PER_CENTER = 6
    RESIDUAL_PROBE_STRIDE = 4

else:  # STAGE 3
    AOA_LIST = [5.0]
    RE_LIST = [1e5]

    # Stage 3 is optimization, not large retraining
    N_INT = 1500
    N_NEAR = 300
    N_WAKE = 300
    N_AIRFOIL = 400
    N_INLET = 250
    N_OUTLET = 250
    N_TOP = 250
    N_BOT = 250

    ADAM_STEPS = 0
    LBFGS_STEPS = 0
    ADAPTIVE_EVERY = 0
    ADAPTIVE_CAP = 0
    CANDIDATE_POOL = 0
    GAUSS_PER_CENTER = 0
    RESIDUAL_PROBE_STRIDE = 1

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

PARSEC_OPT_THRESHOLD = 4.5

In [ ]:
# ============================================================================
# CELL 4 ─ Build train/val/test airfoil library
# ============================================================================
def load_airfoil_library(folder: Path, n_boundary: int = 1200):
    lib = []
    skipped = []

    for f in sorted(folder.glob("*.dat")):
        try:
            geom = AirfoilGeometry.from_dat(f, n_boundary=n_boundary)

            parsec_params = None
            fit_report = None
            fit_mse = None
            use_for_optimization = False

            try:
                pre = preprocess_airfoil_for_parsec(geom.boundary, n_surface=300)
                params, fit_mse = fit_parsec_to_dat(str(f), verbose=False)
                fit_report = fit_quality_report(pre["upper"], pre["lower"], params)

                if fit_report["max_error_pct"] <= PARSEC_OPT_THRESHOLD:
                    parsec_params = params
                    use_for_optimization = True

            except Exception as e_fit:
                fit_report = {
                    "quality": "POOR",
                    "max_error_pct": 100.0,
                    "mean_error_pct": 100.0,
                    "error_message": str(e_fit),
                }

            lib.append(
                {
                    "name": f.stem,
                    "path": str(f),
                    "geometry": geom,
                    "parsec_params": parsec_params,
                    "fit_report": fit_report,
                    "fit_mse": fit_mse,
                    "use_for_optimization": use_for_optimization,
                }
            )

        except Exception as e:
            skipped.append((f.name, str(e)))

    return lib, skipped


TRAIN_AIRFOILS, TRAIN_SKIPPED = load_airfoil_library(TRAIN_DIR)
VAL_AIRFOILS, VAL_SKIPPED = load_airfoil_library(VAL_DIR)
TEST_AIRFOILS, TEST_SKIPPED = load_airfoil_library(TEST_DIR)

print(f"Loaded {len(TRAIN_AIRFOILS)} training airfoils")
print(f"Loaded {len(VAL_AIRFOILS)} validation airfoils")
print(f"Loaded {len(TEST_AIRFOILS)} test airfoils")

if TRAIN_SKIPPED or VAL_SKIPPED or TEST_SKIPPED:
    print("\nSkipped files:")
    for item in TRAIN_SKIPPED + VAL_SKIPPED + TEST_SKIPPED:
        print(" ", item)

Loaded 28 training airfoils
Loaded 4 validation airfoils
Loaded 4 test airfoils


In [ ]:
# ============================================================================
# DEVICE SETUP
# ============================================================================
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# ============================================================================
# CELL 5 ─ Model create or resume
# ============================================================================
model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

# if RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists():
#     model = load_checkpoint(str(RESUME_CHECKPOINT), device=device)
#     print("Resumed model from:", RESUME_CHECKPOINT)

print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))



MLP(5→4, 64×16, act=silu, params=64,068)
Total parameters: 64068


In [ ]:
# # ============================================================================
# # CELL 6 ─ Train stages 1/2
# # ============================================================================
# history = []
# adaptive_weights = None

# if STAGE in (1, 2):
#     print("=" * 62)
#     print(" STAGE 1: Pilot" if STAGE == 1 else " STAGE 2: Research")
#     print(f" AdamW steps = {ADAM_STEPS}")
#     print(f" L-BFGS steps = {LBFGS_STEPS}")
#     print(f" Adaptive every = {ADAPTIVE_EVERY}")
#     print(f" Adaptive cap   = {ADAPTIVE_CAP}")
#     print("=" * 62)

#     model, adaptive_weights, history = train_multi_airfoil(
#         model=model,
#         train_airfoils=TRAIN_AIRFOILS,
#         val_airfoils=VAL_AIRFOILS,
#         aoa_list=AOA_LIST,
#         re_list=RE_LIST,
#         xlim=XLIM,
#         ylim=YLIM,
#         N_int=N_INT,
#         N_near=N_NEAR,
#         N_airfoil=N_AIRFOIL,
#         N_inlet=N_INLET,
#         N_outlet=N_OUTLET,
#         N_top=N_TOP,
#         N_bot=N_BOT,
#         N_wake=N_WAKE,
#         near_band=NEAR_BAND,
#         adam_steps=ADAM_STEPS,
#         lr_adam=LR_ADAM,
#         lr_min=LR_MIN,
#         lr_lambda=LR_LAMBDA,
#         weight_decay=WEIGHT_DECAY,
#         refresh_every=ADAPTIVE_EVERY,
#         adaptive_every=ADAPTIVE_EVERY,
#         adaptive_cap=ADAPTIVE_CAP,
#         candidate_pool=CANDIDATE_POOL,
#         gauss_per_center=GAUSS_PER_CENTER,
#         residual_probe_stride=RESIDUAL_PROBE_STRIDE,
#         lbfgs_steps=LBFGS_STEPS,
#         print_every=PRINT_EVERY,
#         save_every=SAVE_EVERY,
#         val_every=VAL_EVERY,
#         out_dir=str(CKPT_DIR),
#         device=device,
#         seed=42,
#         resume_checkpoint=str(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None,
#     )

#     print("Training finished.")
#     if adaptive_weights is not None:
#         print("Final adaptive weights:", adaptive_weights.all_weights())

#     save_checkpoint(model, str(CKPT_DIR / "final_model.pt"))


In [ ]:
import importlib
import train_loop


importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [ ]:
# ============================================================================
# HIGH-Re 100-COMBINATION CONFIG
# ============================================================================

STAGE = 2

# 25 airfoils × (2 AoA × 2 Re) = 100 combinations
AOA_LIST = [0.0, 4.0]
RE_LIST  = [1.0e6, 2.0e6]

# research-lite but high-Re safe config
N_INT = 3000
N_NEAR = 600
N_WAKE = 600
N_AIRFOIL = 800
N_INLET = 500
N_OUTLET = 500
N_TOP = 500
N_BOT = 500

ADAM_STEPS = 5000
LBFGS_STEPS = 10

ADAPTIVE_EVERY = 100
ADAPTIVE_CAP = 2000
CANDIDATE_POOL = 6000
GAUSS_PER_CENTER = 6
RESIDUAL_PROBE_STRIDE = 4

XLIM = (-1.0, 2.0)
YLIM = (-1.0, 1.0)
NEAR_BAND = 0.003

LR_ADAM = 3e-4
LR_MIN = 1e-5
LR_LAMBDA = 3e-4
WEIGHT_DECAY = 1e-5

PRINT_EVERY = 100
SAVE_EVERY = 500
VAL_EVERY = 2

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

In [ ]:
import importlib
import train_loop
importlib.reload(train_loop)

from train_loop import train_multi_airfoil
print(train_multi_airfoil)

<function train_multi_airfoil at 0x7e3ecc7c36a0>


In [ ]:
import importlib
import losses
import train_loop

importlib.reload(losses)
importlib.reload(train_loop)

<module 'train_loop' from '/content/train_loop.py'>

In [ ]:
from pathlib import Path
import json

state_file = Path(OUT_DIR) / "combo_state.json"
state = json.loads(state_file.read_text())
print(state)

{'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [ ]:
from pathlib import Path
import json

state_file = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
state = json.loads(state_file.read_text())

state["next_block_index"] = 90
state["finished_all_selected_combos"] = False

state_file.write_text(json.dumps(state, indent=2))
print("combo_state.json fixed for resume from human block 91")
print(state)

combo_state.json fixed for resume from human block 91
{'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
from pathlib import Path
import json
import torch

STATE_FILE = Path("/content/drive/MyDrive/AeroPINN/combo_state.json")
RESUME_CHECKPOINT = Path("/content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt")
OUT_DIR = Path("/content/drive/MyDrive/AeroPINN")

print("Checkpoint exists:", RESUME_CHECKPOINT.exists())
print("State exists:", STATE_FILE.exists())

state = json.loads(STATE_FILE.read_text())
state["next_block_index"] = 90
state["finished_all_selected_combos"] = False
STATE_FILE.write_text(json.dumps(state, indent=2))
print("Updated state:", state)

ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
print("Checkpoint keys:", ckpt.keys())

model.load_state_dict(ckpt["model_state"])
print("Loaded model_state from:", RESUME_CHECKPOINT)

history = ckpt.get("history", {})
print("Saved step:", ckpt.get("step"))
print("Saved block_index:", ckpt.get("block_index"))

Checkpoint exists: True
State exists: True
Updated state: {'next_block_index': 90, 'total_selected_combos': 100, 'completed_blocks_this_run': 10, 'finished_all_selected_combos': False}
Checkpoint keys: dict_keys(['step', 'block_index', 'model_state', 'optimizer_state', 'scheduler_state', 'lambda_optimizer_state', 'loss_weights_state', 'adaptive_sampler_state', 'history', 'extra'])
Loaded model_state from: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
Saved step: 5000
Saved block_index: 90


In [ ]:
# ============================================================================
# CELL ─ High-Re run settings
# ============================================================================

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100

print("BLOCK_STEPS =", BLOCK_STEPS)
print("BLOCKS_PER_RUN =", BLOCKS_PER_RUN)
print("MAX_TOTAL_COMBOS =", MAX_TOTAL_COMBOS)

BLOCK_STEPS = 500
BLOCKS_PER_RUN = 10
MAX_TOTAL_COMBOS = 100


In [ ]:
# Check which required variables are still missing
required_names = [
    "TRAIN_AIRFOILS", "VAL_AIRFOILS",
    "AOA_LIST", "RE_LIST",
    "XLIM", "YLIM",
    "N_INT", "N_NEAR", "N_AIRFOIL", "N_INLET", "N_OUTLET", "N_TOP", "N_BOT", "N_WAKE",
    "NEAR_BAND",
    "ADAM_STEPS", "LBFGS_STEPS",
    "LR_ADAM", "LR_MIN", "LR_LAMBDA", "WEIGHT_DECAY",
    "ADAPTIVE_EVERY", "ADAPTIVE_CAP",
    "CANDIDATE_POOL", "GAUSS_PER_CENTER", "RESIDUAL_PROBE_STRIDE",
    "PRINT_EVERY", "SAVE_EVERY", "VAL_EVERY",
    "train_multi_airfoil"
]

missing = [name for name in required_names if name not in globals()]
print("Missing:", missing)

Missing: []


In [ ]:
# ============================================================================
# CELL ─ Run 10 new high-Re combinations
# ============================================================================

print("=" * 62)
print(" STAGE 2: High-Re 100-combination training")
print(f" AdamW steps = {ADAM_STEPS}")
print(f" L-BFGS steps = {LBFGS_STEPS}")
print(f" Adaptive every = {ADAPTIVE_EVERY}")
print(f" Adaptive cap   = {ADAPTIVE_CAP}")
print(f" Blocks/run     = {10}")
print(f" Max combos     = {100}")
print(f" AoA list       = {AOA_LIST}")
print(f" Re list        = {RE_LIST}")
print("=" * 62)

model, adaptive_weights, history = train_multi_airfoil(
    model=model,
    train_airfoils=TRAIN_AIRFOILS,
    val_airfoils=VAL_AIRFOILS,
    aoa_list=[0.0, 4.0],
    re_list=[1e6, 2e6],
    xlim=XLIM,
    ylim=YLIM,
    N_int=N_INT,
    N_near=N_NEAR,
    N_airfoil=N_AIRFOIL,
    N_inlet=N_INLET,
    N_outlet=N_OUTLET,
    N_top=N_TOP,
    N_bot=N_BOT,
    N_wake=N_WAKE,
    near_band=NEAR_BAND,
    adam_steps=ADAM_STEPS,
    lr_adam=LR_ADAM,
    lr_min=LR_MIN,
    lr_lambda=LR_LAMBDA,
    weight_decay=WEIGHT_DECAY,
    refresh_every=ADAPTIVE_EVERY,
    adaptive_every=ADAPTIVE_EVERY,
    adaptive_cap=ADAPTIVE_CAP,
    candidate_pool=CANDIDATE_POOL,
    gauss_per_center=GAUSS_PER_CENTER,
    residual_probe_stride=RESIDUAL_PROBE_STRIDE,
    lbfgs_steps=LBFGS_STEPS,
    print_every=PRINT_EVERY,
    save_every=SAVE_EVERY,
    val_every=VAL_EVERY,
    out_dir=Path("/content/drive/MyDrive/AeroPINN"),
    device=device,
    seed=42,
    block_steps=500,
    blocks_per_run=10,
    max_total_combos=100,
    resume_checkpoint=str(RESUME_CHECKPOINT),
)

print("\nBatch finished.")
if adaptive_weights is not None:
    print("Final adaptive weights:", adaptive_weights.all_weights())
print("History rows:", len(history))

 STAGE 2: High-Re 100-combination training
 AdamW steps = 5000
 L-BFGS steps = 10
 Adaptive every = 100
 Adaptive cap   = 2000
 Blocks/run     = 10
 Max combos     = 100
 AoA list       = [0.0, 4.0]
 Re list        = [1000000.0, 2000000.0]
train_multi_airfoil: 28 airfoils, 4 conditions, selected combos=100, running blocks 90..99

BLOCK 91/100
  Airfoil : naca4412
  AoA     : 0.0 deg
  Re      : 1.000e+06
  Steps   : 500
  step   100  loss=1.1719e-06  pde=1.2880e-06  wall=5.6693e-09  lr=1.52e-04  λwall=0.41  λpde=0.84
  step   200  loss=1.8803e-06  pde=9.8623e-07  wall=4.1450e-09  lr=7.70e-05  λwall=0.13  λpde=1.88
  step   300  loss=1.6351e-06  pde=7.5386e-07  wall=4.0259e-09  lr=3.90e-05  λwall=0.13  λpde=2.13
  step   400  loss=1.5146e-06  pde=6.9703e-07  wall=3.9732e-09  lr=1.97e-05  λwall=0.13  λpde=2.13
  step   500  loss=1.6220e-06  pde=7.4748e-07  wall=4.0744e-09  lr=1.00e-05  λwall=0.13  λpde=2.13

BLOCK 92/100
  Airfoil : naca0015
  AoA     : 0.0 deg
  Re      : 2.000e+06
  St

In [ ]:
import json
from pathlib import Path
import torch

def save_batch_checkpoint_permanent(
    model,
    adaptive_weights,
    history,
    out_dir,
    next_block_index,
    total_selected_combos,
    tag="batch_checkpoint",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = out_dir / f"{tag}.pt"
    state_path = out_dir / "combo_state.json"
    history_path = out_dir / "history.json"

    payload = {
        "model_state": model.state_dict(),
        "loss_weights_state": adaptive_weights.state_dict() if adaptive_weights is not None else None,
        "block_index": int(next_block_index),
        "history": history,
        "extra": {
            "next_block_index": int(next_block_index),
            "total_selected_combos": int(total_selected_combos),
        },
    }
    torch.save(payload, ckpt_path)

    state_path.write_text(
        json.dumps(
            {
                "next_block_index": int(next_block_index),
                "total_selected_combos": int(total_selected_combos),
                "finished_all_selected_combos": bool(next_block_index >= total_selected_combos),
            },
            indent=2,
        )
    )

    history_path.write_text(json.dumps(history, indent=2))

    print("Saved permanent checkpoint to:", ckpt_path)
    print("Saved combo state to        :", state_path)
    print("Saved history to            :", history_path)

In [ ]:
#load it later

# import torch
# from pathlib import Path

# ckpt_path = Path(OUT_DIR) / "multi_airfoil_last.pt"
# data = torch.load(ckpt_path, map_location=device, weights_only=False)

# model.load_state_dict(data["model_state"])

# if adaptive_weights is not None and data.get("loss_weights_state") is not None:
#     adaptive_weights.load_state_dict(data["loss_weights_state"])

# history = data.get("history", [])
# next_block_index = data.get("block_index", 0)

# print("Loaded checkpoint from:", ckpt_path)
# print("Next block index:", next_block_index)
# print("History rows:", len(history))

### **Validation**

In [55]:
import importlib
import losses
import train_loop
import postprocess

importlib.reload(losses)
importlib.reload(train_loop)
importlib.reload(postprocess)

<module 'postprocess' from '/content/postprocess.py'>

In [56]:
from pathlib import Path
import torch
from network import MLP

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

ckpt_path = Path("/content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt")
data = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(data["model_state"])
model.eval()

history = data.get("history", [])
print("Loaded checkpoint:", ckpt_path)
print("History rows:", len(history))

Loaded checkpoint: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
History rows: 100


In [57]:
# ============================================================================
# CELL 7 ─ Validation + reports
# ============================================================================
from pathlib import Path

if STAGE in (1, 2):
    VAL_STAGE_DIR = Path("/content/drive/MyDrive/AeroPINN/validation_outputs") / f"stage_{STAGE}"
    VAL_STAGE_DIR.mkdir(parents=True, exist_ok=True)

    all_reports = {}

    # ------------------------------------------------------------
    # Validation library
    # ------------------------------------------------------------
    if len(VAL_AIRFOILS) > 0:
        df_val, val_summary = validate_model_on_library(
            model=model,
            airfoil_library=VAL_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir="/content/drive/MyDrive/AeroPINN",
            device=device,
            max_cases=24,
        )
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)
    else:
        df_val = None
        val_summary = {"message": "No validation airfoils available"}
        all_reports["validation_summary"] = val_summary
        print("Validation summary:", val_summary)

    # ------------------------------------------------------------
    # Test library (optional but recommended)
    # ------------------------------------------------------------
    if "TEST_AIRFOILS" in globals() and len(TEST_AIRFOILS) > 0:
        df_test, test_summary = validate_model_on_library(
            model=model,
            airfoil_library=TEST_AIRFOILS,
            aoa_list=AOA_LIST,
            re_list=RE_LIST,
            out_dir=VAL_STAGE_DIR / "test",
            device=device,
            max_cases=24,
        )
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)
    else:
        test_summary = {"message": "No test airfoils available"}
        all_reports["test_summary"] = test_summary
        print("Test summary:", test_summary)

    # ------------------------------------------------------------
    # Export run report
    # ------------------------------------------------------------
    run_report = export_run_report(
        history=history if "history" in globals() else [],
        extra={
            "stage": STAGE,
            "aoa_list": list(AOA_LIST),
            "re_list": list(RE_LIST),
            "validation_summary": val_summary,
            "test_summary": test_summary,
            "n_train_airfoils": len(TRAIN_AIRFOILS) if "TRAIN_AIRFOILS" in globals() else 0,
            "n_val_airfoils": len(VAL_AIRFOILS) if "VAL_AIRFOILS" in globals() else 0,
            "n_test_airfoils": len(TEST_AIRFOILS) if "TEST_AIRFOILS" in globals() else 0,
        },
        out_dir=Path(OUT_DIR) / "run_report",
    )

    print("Run report:", run_report)

In [58]:
import pandas as pd
import numpy as np

def robust_ld_summary(csv_path, cd_threshold=1e-3):
    df = pd.read_csv(csv_path)

    summary = {
        "n_cases": len(df),
        "mean_CL": float(df["CL"].mean()),
        "mean_CD": float(df["CD"].mean()),
        "median_LD": float(df["LD"].median()),
    }

    df_pos = df[df["CL"] > 0]
    summary["n_positive_CL"] = int(len(df_pos))
    summary["mean_LD_positive_CL"] = float(df_pos["LD"].mean()) if len(df_pos) else np.nan

    df_cd = df[df["CD"] > cd_threshold]
    summary["n_cd_gt_threshold"] = int(len(df_cd))
    summary["mean_LD_cd_gt_threshold"] = float(df_cd["LD"].mean()) if len(df_cd) else np.nan

    df_good = df[(df["CL"] > 0) & (df["CD"] > cd_threshold)]
    summary["n_good_cases"] = int(len(df_good))
    summary["mean_LD_good_cases"] = float(df_good["LD"].mean()) if len(df_good) else np.nan

    return summary, df

In [59]:
import pandas as pd
import numpy as np

def robust_ld_summary(csv_path, cd_threshold=1e-3):
    df = pd.read_csv(csv_path)

    summary = {
        "n_cases": len(df),
        "mean_CL": float(df["CL"].mean()),
        "mean_CD": float(df["CD"].mean()),
        "median_LD": float(df["LD"].median()),
    }

    df_pos = df[df["CL"] > 0]
    summary["n_positive_CL"] = int(len(df_pos))
    summary["mean_LD_positive_CL"] = float(df_pos["LD"].mean()) if len(df_pos) else np.nan

    df_cd = df[df["CD"] > cd_threshold]
    summary["n_cd_gt_threshold"] = int(len(df_cd))
    summary["mean_LD_cd_gt_threshold"] = float(df_cd["LD"].mean()) if len(df_cd) else np.nan

    df_good = df[(df["CL"] > 0) & (df["CD"] > cd_threshold)]
    summary["n_good_cases"] = int(len(df_good))
    summary["mean_LD_good_cases"] = float(df_good["LD"].mean()) if len(df_good) else np.nan

    return summary, df

In [ ]:
# val_summary_robust, df_val = robust_ld_summary(
#     "/content/drive/MyDrive/AeroPINN/pilot_fix_run/validation/stage_2/validation/validation_metrics.csv"
# )

# test_summary_robust, df_test = robust_ld_summary(
#     "/content/drive/MyDrive/AeroPINN/validation_outputs/stage_2/test/validation_metrics.csv"
# )

# #
# print("Robust TEST summary:", test_summary_robust)print("Robust VAL summary:", val_summary_robust)

**debugging**


In [60]:
import importlib
import losses
import train_loop
import postprocess

importlib.reload(losses)
importlib.reload(train_loop)
importlib.reload(postprocess)

<module 'postprocess' from '/content/postprocess.py'>

In [61]:
# ============================================================================
# SANITY CHECK BEFORE STAGE 3
# ============================================================================
from postprocess import compute_lift_drag
import pandas as pd

def quick_case_check(model, airfoil_items, aoa_list, re_list, device="cpu"):
    rows = []
    for item in airfoil_items:
        for aoa in aoa_list:
            for re in re_list:
                m = compute_lift_drag(
                    model=model,
                    geometry=item["geometry"],
                    alpha_deg=float(aoa),
                    Re_phys=float(re),
                    device=device,
                )
                rows.append({
                    "airfoil": item["name"],
                    "alpha_deg": float(aoa),
                    "Re_phys": float(re),
                    "CL": m["CL"],
                    "CD": m["CD"],
                    "LD": m["LD"],
                })
    df = pd.DataFrame(rows)
    return df

# choose a few likely stable airfoils first
demo_names = ["eppler186", "mh60", "mh78", "naca2412"]
demo_items = [a for a in (TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS) if a["name"] in demo_names]

df_demo = quick_case_check(
    model=model,
    airfoil_items=demo_items,
    aoa_list=[4.0],
    re_list=[1.0e6, 2.0e6],
    device=device,
)

print(df_demo)

     airfoil  alpha_deg    Re_phys        CL        CD        LD
0  eppler186        4.0  1000000.0 -0.002866  0.001800 -1.592013
1  eppler186        4.0  2000000.0 -0.018871  0.029913 -0.630870
2       mh60        4.0  1000000.0 -0.014813  0.025170 -0.588512
3       mh60        4.0  2000000.0  0.028327  0.062061  0.456437
4       mh78        4.0  1000000.0  0.017047  0.008318  2.049349
5       mh78        4.0  2000000.0 -0.035075  0.218526 -0.160506
6   naca2412        4.0  1000000.0 -0.010958  0.026284 -0.416899
7   naca2412        4.0  2000000.0  0.335788  0.061072  5.498238


### **gate**

In [62]:
# ============================================================================
# STAGE 3 SAFETY GATE
# ============================================================================

from postprocess import compute_lift_drag

def stage3_case_gate(
    model,
    target_item,
    alpha_deg,
    Re_phys,
    device="cpu",
    min_cd=1e-3,
):
    metrics = compute_lift_drag(
        model=model,
        geometry=target_item["geometry"],
        alpha_deg=float(alpha_deg),
        Re_phys=float(Re_phys),
        device=device,
    )

    ok = True
    reasons = []

    if target_item.get("parsec_params", None) is None:
        ok = False
        reasons.append("No acceptable PARSEC fit")

    if not np.isfinite(metrics["CL"]) or not np.isfinite(metrics["CD"]) or not np.isfinite(metrics["LD"]):
        ok = False
        reasons.append("Non-finite aerodynamic metrics")

    if metrics["CD"] <= min_cd:
        ok = False
        reasons.append(f"CD too small ({metrics['CD']:.3e})")

    if metrics["CL"] <= 0:
        ok = False
        reasons.append(f"CL not positive ({metrics['CL']:.6f})")

    if metrics["LD"] <= 0:
        ok = False
        reasons.append(f"L/D not positive ({metrics['LD']:.6f})")

    return {
        "ok": ok,
        "metrics": metrics,
        "reasons": reasons,
    }

In [63]:
# ============================================================================
# STAGE 3 SAFETY GATE
# ============================================================================

from postprocess import compute_lift_drag

def stage3_case_gate(
    model,
    target_item,
    alpha_deg,
    Re_phys,
    device="cpu",
    min_cd=1e-3,
):
    metrics = compute_lift_drag(
        model=model,
        geometry=target_item["geometry"],
        alpha_deg=float(alpha_deg),
        Re_phys=float(Re_phys),
        device=device,
    )

    ok = True
    reasons = []

    if target_item.get("parsec_params", None) is None:
        ok = False
        reasons.append("No acceptable PARSEC fit")

    if not np.isfinite(metrics["CL"]) or not np.isfinite(metrics["CD"]) or not np.isfinite(metrics["LD"]):
        ok = False
        reasons.append("Non-finite aerodynamic metrics")

    if metrics["CD"] <= min_cd:
        ok = False
        reasons.append(f"CD too small ({metrics['CD']:.3e})")

    if metrics["CL"] <= 0:
        ok = False
        reasons.append(f"CL not positive ({metrics['CL']:.6f})")

    if metrics["LD"] <= 0:
        ok = False
        reasons.append(f"L/D not positive ({metrics['LD']:.6f})")

    return {
        "ok": ok,
        "metrics": metrics,
        "reasons": reasons,
    }

In [64]:
# ============================================================================
# FIND STABLE STAGE-3 CANDIDATES
# ============================================================================

def find_stage3_candidates(all_airfoils, model, aoa_list, re_list, device="cpu"):
    rows = []
    for item in all_airfoils:
        for aoa in aoa_list:
            for re in re_list:
                gate = stage3_case_gate(
                    model=model,
                    target_item=item,
                    alpha_deg=aoa,
                    Re_phys=re,
                    device=device,
                )
                rows.append({
                    "airfoil": item["name"],
                    "alpha_deg": float(aoa),
                    "Re_phys": float(re),
                    "ok": gate["ok"],
                    "CL": gate["metrics"]["CL"],
                    "CD": gate["metrics"]["CD"],
                    "LD": gate["metrics"]["LD"],
                    "reasons": "; ".join(gate["reasons"]),
                })
    return pd.DataFrame(rows)

all_airfoils = TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS
df_stage3 = find_stage3_candidates(
    all_airfoils=all_airfoils,
    model=model,
    aoa_list=[4.0],
    re_list=[1.0e6, 2.0e6],
    device=device,
)

print(df_stage3[df_stage3["ok"]].sort_values("LD", ascending=False).head(20))

        airfoil  alpha_deg    Re_phys    ok        CL        CD        LD  \
11     naca0009        4.0  2000000.0  True  0.476606  0.058953  8.084504   
9      naca0006        4.0  2000000.0  True  0.339413  0.044186  7.681438   
67     naca0010        4.0  2000000.0  True  0.463860  0.061120  7.589323   
41   naca64_012        4.0  2000000.0  True  0.456333  0.065703  6.945342   
37   naca63_012        4.0  2000000.0  True  0.416753  0.060756  6.859499   
29     naca2412        4.0  2000000.0  True  0.335788  0.061072  5.498238   
31     naca2415        4.0  2000000.0  True  0.416942  0.085039  4.902956   
57     naca0011        4.0  2000000.0  True  0.372217  0.079118  4.704565   
33     naca4412        4.0  2000000.0  True  0.287471  0.070315  4.088303   
13  naca0012-64        4.0  2000000.0  True  0.255134  0.068076  3.747778   
17     naca0018        4.0  2000000.0  True  0.230131  0.094963  2.423376   
8      naca0006        4.0  1000000.0  True  0.020318  0.008594  2.364277   

FINAL

In [66]:
df_stage3[df_stage3["ok"]].sort_values("LD", ascending=False).head(20)

,airfoil,alpha_deg,Re_phys,ok,CL,CD,LD,reasons
11,naca0009,4.0,2000000.0,True,0.476606,0.058953,8.084504,
9,naca0006,4.0,2000000.0,True,0.339413,0.044186,7.681438,
67,naca0010,4.0,2000000.0,True,0.463860,0.061120,7.589323,
41,naca64_012,4.0,2000000.0,True,0.456333,0.065703,6.945342,
37,naca63_012,4.0,2000000.0,True,0.416753,0.060756,6.859499,
29,naca2412,4.0,2000000.0,True,0.335788,0.061072,5.498238,
31,naca2415,4.0,2000000.0,True,0.416942,0.085039,4.902956,
57,naca0011,4.0,2000000.0,True,0.372217,0.079118,4.704565,
33,naca4412,4.0,2000000.0,True,0.287471,0.070315,4.088303,
13,naca0012-64,4.0,2000000.0,True,0.255134,0.068076,3.747778,


recover

In [ ]:
import importlib
import train_loop
import xai

importlib.reload(train_loop)
importlib.reload(xai)

<module 'xai' from '/content/xai.py'>

In [ ]:
# ============================================================================
# RECOVER MODEL FROM SAVED CHECKPOINT
# ============================================================================

from pathlib import Path
import torch
from network import MLP

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = MLP(
    in_dim=5,
    out_dim=4,
    hidden_dim=64,
    num_hidden_layers=16,
    activation="silu",
).to(device)

ckpt_path = Path(OUT_DIR) / "multi_airfoil_last.pt"
print("Checkpoint exists:", ckpt_path.exists())

data = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(data["model_state"])
model.eval()

history = data.get("history", [])
print("Loaded checkpoint:", ckpt_path)
print("History rows:", len(history))
print("Saved block index:", data.get("block_index", None))

Device: cuda
Checkpoint exists: True
Loaded checkpoint: /content/drive/MyDrive/AeroPINN/multi_airfoil_last.pt
History rows: 100
Saved block index: 100


In [ ]:
# ============================================================================
# SAFE STAGE 3 RUN + XAI GENERATION
# ============================================================================

from pathlib import Path
import gc
import torch
import json

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()

TARGET_AIRFOIL_NAME = "naca2412"
OPT_ALPHA = 4.0
OPT_RE = 2.0e6
OPT_ITERS = 30
OPT_MODE = "lbfgs"

all_airfoils = TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS
matches = [a for a in all_airfoils if a["name"] == TARGET_AIRFOIL_NAME]
if not matches:
    raise ValueError(f"Airfoil '{TARGET_AIRFOIL_NAME}' not found")

target = matches[0]
if target["parsec_params"] is None:
    raise ValueError(
        f"Airfoil '{TARGET_AIRFOIL_NAME}' does not have acceptable PARSEC fit"
    )

# baseline gate
gate = stage3_case_gate(
    model=model,
    target_item=target,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    device=device,
)

print("Baseline metrics:", gate["metrics"])
print("Gate reasons:", gate["reasons"])

if not gate["ok"]:
    raise ValueError(
        f"Stage 3 blocked for {TARGET_AIRFOIL_NAME} at AoA={OPT_ALPHA}, Re={OPT_RE}. "
        f"Reasons: {gate['reasons']}"
    )

# output folder
case_dir = Path(OPT_OUT_DIR) / f"{TARGET_AIRFOIL_NAME}_a{int(OPT_ALPHA)}_Re{int(OPT_RE)}"
case_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("STAGE 3 ─ FINAL OPTIMIZATION")
print(f"Target airfoil : {TARGET_AIRFOIL_NAME}")
print(f"AoA            : {OPT_ALPHA} deg")
print(f"Re             : {OPT_RE:.3e}")
print(f"Mode           : {OPT_MODE}")
print(f"Iterations     : {OPT_ITERS}")
print(f"Output folder  : {case_dir}")
print("=" * 70)

# optimize
best_params, best_info = optimize_shape(
    model=model,
    init_parsec=target["parsec_params"],
    mode=OPT_MODE,
    iters=OPT_ITERS,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    xlim=XLIM,
    ylim=YLIM,
    N_int=N_INT,
    N_near=N_NEAR,
    N_airfoil=N_AIRFOIL,
    N_inlet=N_INLET,
    N_outlet=N_OUTLET,
    N_top=N_TOP,
    N_bot=N_BOT,
    N_wake=N_WAKE,
    device=device,
)

print("Optimization best_info:", best_info)

# export optimized package
exported = export_optimized_airfoil(
    original_geometry=target["geometry"],
    fitted_parsec_params=target["parsec_params"],
    optimized_parsec_params=best_params,
    best_info=best_info,
    out_dir=case_dir,
)
print("Optimized export:", exported)

# XAI
xai_dir = case_dir / "xai"
xai_dir.mkdir(parents=True, exist_ok=True)

xai_report = run_xai_suite(
    model=model,
    geometry=target["geometry"],
    parsec_params=best_params,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    xlim=XLIM,
    ylim=YLIM,
    loss_weights=adaptive_weights.all_weights() if "adaptive_weights" in globals() and adaptive_weights is not None else None,
    out_dir=xai_dir,
    device=device,
)

print("\nXAI generation finished.")
print("XAI dir:", xai_dir)
print("PREM exists:", (xai_dir / "prem_residual_map.png").exists())
print("FFSV exists:", (xai_dir / "ffsv_saliency.png").exists())
print("Report exists:", (xai_dir / "xai_report.json").exists())

if (xai_dir / "xai_report.json").exists():
    report = json.loads((xai_dir / "xai_report.json").read_text())
    print("XAI report keys:", list(report.keys()))

print("\nStage 3 complete.")

Baseline metrics: {'CL': 0.33578837768987146, 'CD': 0.06107199304056003, 'CD_signed': 0.06107199304056003, 'LD': 5.49823840638151, 'boundary': {'x': array([1.00000000e+00, 9.99406874e-01, 9.98517215e-01, 9.95420575e-01,
       9.89965141e-01, 9.86328185e-01, 9.80872750e-01, 9.77235794e-01,
       9.71780360e-01, 9.68143404e-01, 9.62687969e-01, 9.59051013e-01,
       9.53595579e-01, 9.49981511e-01, 9.48959887e-01, 9.48303878e-01,
       9.42918003e-01, 9.39281046e-01, 9.33825612e-01, 9.30188656e-01,
       9.24733222e-01, 9.21096265e-01, 9.15640831e-01, 9.12003875e-01,
       9.06548381e-01, 9.02911425e-01, 8.99558187e-01, 8.99196386e-01,
       8.98653746e-01, 8.98248613e-01, 8.93800080e-01, 8.90163124e-01,
       8.84707689e-01, 8.81070733e-01, 8.75615299e-01, 8.71978343e-01,
       8.66522908e-01, 8.62885952e-01, 8.57430518e-01, 8.53793561e-01,
       8.48338127e-01, 8.44701171e-01, 8.39245737e-01, 8.35608780e-01,
       8.30153346e-01, 8.26516390e-01, 8.21060956e-01, 8.17423999e-01,

AttributeError: 'NoneType' object has no attribute 'detach'

### **fixes**

In [81]:
# ============================================================================
# CELL 1 ─ Patch xai module aliases
# ============================================================================

import importlib
import xai

importlib.reload(xai)

# map the actual functions to the names used in run_xai_suite
xai.prem_residual_map = xai.plot_residual_maps
xai.ffsv_saliency_map = xai.flow_field_saliency

from xai import run_xai_suite

print("Patched xai aliases:")
print("prem_residual_map ->", xai.prem_residual_map.__name__)
print("ffsv_saliency_map ->", xai.ffsv_saliency_map.__name__)

Patched xai aliases:
prem_residual_map -> plot_residual_maps
ffsv_saliency_map -> flow_field_saliency


In [83]:
# ============================================================================
# CELL 3 ─ Corrected XAI generation
# ============================================================================

from pathlib import Path
import json

TARGET_AIRFOIL_NAME = "naca2412"
OPT_ALPHA = 4.0
OPT_RE = 2.0e6

all_airfoils = TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS
target = [a for a in all_airfoils if a["name"] == TARGET_AIRFOIL_NAME][0]

case_dir = Path(OPT_OUT_DIR) / f"{TARGET_AIRFOIL_NAME}_a{int(OPT_ALPHA)}_Re{int(OPT_RE)}"
xai_dir = case_dir / "xai"
xai_dir.mkdir(parents=True, exist_ok=True)

xai_report = run_xai_suite(
    model=model,
    geometry=target["geometry"],
    parsec_params=best_params,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    xlim=XLIM,
    ylim=YLIM,
    loss_weights=adaptive_weights.all_weights() if "adaptive_weights" in globals() and adaptive_weights is not None else None,
    out_dir=xai_dir,
    device=device,
)

print("XAI dir:", xai_dir)
print("PREM exists  :", (xai_dir / "prem_residual_map.png").exists())
print("FFSV exists  :", (xai_dir / "ffsv_saliency.png").exists())
print("Report exists:", (xai_dir / "xai_report.json").exists())

if (xai_dir / "xai_report.json").exists():
    report = json.loads((xai_dir / "xai_report.json").read_text())
    print("XAI report keys:", list(report.keys()))
    print("FFSV report entry:", report.get("FFSV", {}))

XAI dir: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000/xai
PREM exists  : True
FFSV exists  : True
Report exists: True
XAI report keys: ['PREM', 'GAM', 'IGIE', 'FFSV', 'LWSA']
FFSV report entry: {'error': 'TypeError("flow_field_saliency() got an unexpected keyword argument \'out_path\'")', 'traceback': 'Traceback (most recent call last):\n  File "/content/xai.py", line 375, in run_xai_suite\n    ffsv = ffsv_saliency_map(\n           ^^^^^^^^^^^^^^^^^^\nTypeError: flow_field_saliency() got an unexpected keyword argument \'out_path\'\n'}


In [ ]:
# ============================================================================
# CELL 4 ─ XAI check
# ============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

case_dir = Path(OPT_OUT_DIR) / "naca2412_a4_Re2000000"
xai_dir = case_dir / "xai"
report_path = xai_dir / "xai_report.json"

print("Using case folder:", case_dir)
print("Exists PREM PNG :", (xai_dir / "prem_residual_map.png").exists())
print("Exists FFSV PNG :", (xai_dir / "ffsv_saliency.png").exists())
print("Exists report   :", report_path.exists())

if not report_path.exists():
    raise FileNotFoundError(f"Missing xai_report.json at {report_path}")

report = json.loads(report_path.read_text())

param_names = [
    "r_le", "x_up", "y_up", "yxx_up", "x_lo", "y_lo",
    "yxx_lo", "y_te", "dyt", "alpha_te", "beta_te"
]

gam_abs = np.array(report.get("GAM", {}).get("saliency_abs", [0]*11), dtype=float)
ig_abs = np.array(report.get("IGIE", {}).get("abs_integrated_gradients", [0]*11), dtype=float)

df_xai = pd.DataFrame({
    "param": param_names,
    "GAM_abs": gam_abs,
    "IG_abs": ig_abs,
})

df_xai["GAM_rank"] = df_xai["GAM_abs"].rank(ascending=False, method="dense")
df_xai["IG_rank"] = df_xai["IG_abs"].rank(ascending=False, method="dense")

print("\nParameter attribution table:")
print(df_xai.sort_values("GAM_abs", ascending=False))

csv_path = xai_dir / "xai_param_attributions.csv"
df_xai.to_csv(csv_path, index=False)
print("\nSaved attribution CSV:", csv_path)

Using case folder: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000
Exists PREM PNG : True
Exists FFSV PNG : False
Exists report   : True

Parameter attribution table:
       param      GAM_abs  IG_abs  GAM_rank  IG_rank
0       r_le  1038.401554     0.0       1.0      1.0
2       y_up  1012.868617     0.0       2.0      1.0
8        dyt   527.708706     0.0       3.0      1.0
10   beta_te   209.360878     0.0       4.0      1.0
4       x_lo   185.801462     0.0       5.0      1.0
5       y_lo   178.458662     0.0       6.0      1.0
7       y_te   136.829642     0.0       7.0      1.0
3     yxx_up   132.016978     0.0       8.0      1.0
6     yxx_lo   105.906888     0.0       9.0      1.0
1       x_up    62.721347     0.0      10.0      1.0
9   alpha_te     8.877741     0.0      11.0      1.0

Saved attribution CSV: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000/xai/xai_param_attributions.csv


newfix


In [160]:
from pathlib import Path
import os

case_dir = Path(OPT_OUT_DIR) / "naca2412_a4_Re2000000"
xai_dir = case_dir / "xai"
bad_json = xai_dir / "xai_report.json"

if bad_json.exists():
    bad_json.unlink()

print("Removed broken JSON:", bad_json)

Removed broken JSON: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000/xai/xai_report.json


In [161]:
import importlib
import xai

importlib.reload(xai)
from xai import run_xai_suite

In [162]:
from pathlib import Path
import json

TARGET_AIRFOIL_NAME = "naca2412"
OPT_ALPHA = 4.0
OPT_RE = 2.0e6

all_airfoils = TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS
target = [a for a in all_airfoils if a["name"] == TARGET_AIRFOIL_NAME][0]

case_dir = Path(OPT_OUT_DIR) / f"{TARGET_AIRFOIL_NAME}_a{int(OPT_ALPHA)}_Re{int(OPT_RE)}"
xai_dir = case_dir / "xai"
xai_dir.mkdir(parents=True, exist_ok=True)

if "best_params" not in globals():
    opt_json = case_dir / "optimized_parsec.json"
    data = json.loads(opt_json.read_text())
    best_params = data["optimized_parsec"]

xai_report = run_xai_suite(
    model=model,
    geometry=target["geometry"],
    parsec_params=best_params,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    xlim=XLIM,
    ylim=YLIM,
    loss_weights=adaptive_weights.all_weights() if "adaptive_weights" in globals() and adaptive_weights is not None else None,
    out_dir=xai_dir,
    device=device,
)

print("XAI dir:", xai_dir)
print("PREM exists  :", (xai_dir / "prem_residual_map.png").exists())
print("FFSV exists  :", (xai_dir / "ffsv_saliency.png").exists())
print("Report exists:", (xai_dir / "xai_report.json").exists())

report = json.loads((xai_dir / "xai_report.json").read_text())
print("XAI report keys:", list(report.keys()))
print("PREM error:", report.get("PREM", {}).get("error"))
print("FFSV error:", report.get("FFSV", {}).get("error"))
print("IG abs:", report.get("IGIE", {}).get("abs_integrated_gradients"))

XAI dir: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000/xai
PREM exists  : True
FFSV exists  : True
Report exists: True
XAI report keys: ['PREM', 'GAM', 'IGIE', 'FFSV', 'LWSA']
PREM error: None
FFSV error: None
IG abs: [223.97109525486383, 3426.1873212577675, 690.4576829243291, 13114.379053710298, 210.15909141575654, 2322.3232211983877, 234410.89935225117, 11220.309986931163, 1782.846228713933, 242.27178780075778, 16913.099745030453]


In [163]:
import numpy as np
from parsec_fit import PARSEC_BOUNDS

def clip_parsec_inside_bounds(params, margin=1e-8):
    p = np.asarray(params, dtype=float).copy()
    lo = np.array([b[0] for b in PARSEC_BOUNDS], dtype=float) + margin
    hi = np.array([b[1] for b in PARSEC_BOUNDS], dtype=float) - margin
    return np.clip(p, lo, hi).tolist()

In [134]:
from pathlib import Path
import json
import numpy as np

case_dir = Path(OPT_OUT_DIR) / "naca2412_a4_Re2000000"
report = json.loads((case_dir / "xai" / "xai_report.json").read_text())

print("FFSV stats:", report["FFSV"].get("stats", {}))
print("IG abs:", np.array(report["IGIE"]["abs_integrated_gradients"]))

FFSV stats: {}
IG abs: [2.23971095e+02 3.42618732e+03 6.90457683e+02 1.31143791e+04
 2.10159091e+02 2.32232322e+03 2.34410899e+05 1.12203100e+04
 1.78284623e+03 2.42271788e+02 1.69130997e+04]


In [135]:
from pathlib import Path
import json

case_dir = Path(OPT_OUT_DIR) / "naca2412_a4_Re2000000"
xai_dir = case_dir / "xai"
report = json.loads((xai_dir / "xai_report.json").read_text())

print("PREM stats:", report["PREM"].get("stats"))
print("FFSV stats:", report["FFSV"].get("stats"))
print("PREM png:", report["PREM"].get("png"))
print("FFSV png:", report["FFSV"].get("png"))

PREM stats: None
FFSV stats: None
PREM png: None
FFSV png: None


In [136]:
p = Path("/content/xai.py")
txt = p.read_text()

print("has prem wrapper :", "def prem_residual_map" in txt)
print("has ffsv wrapper :", "def ffsv_saliency_map" in txt)
print("has run_xai_suite:", "def run_xai_suite" in txt)
print("has old mean pressure target:", "torch.mean(Y[:, 2:3])" in txt)

has prem wrapper : True
has ffsv wrapper : True
has run_xai_suite: True
has old mean pressure target: False


In [137]:
import sys
import importlib

if "xai" in sys.modules:
    del sys.modules["xai"]

import xai
importlib.reload(xai)

print("Loaded from:", xai.__file__)

Loaded from: /content/xai.py


In [138]:
from pathlib import Path

case_dir = Path(OPT_OUT_DIR) / "naca2412_a4_Re2000000"
xai_dir = case_dir / "xai"
xai_dir.mkdir(parents=True, exist_ok=True)

for fn in ["xai_report.json", "prem_residual_map.png", "ffsv_saliency.png"]:
    fp = xai_dir / fn
    if fp.exists():
        fp.unlink()
        print("Deleted:", fp)

Deleted: /content/drive/MyDrive/AeroPINN/optimized_results/naca2412_a4_Re2000000/xai/xai_report.json


In [139]:
from xai import run_xai_suite
import json

TARGET_AIRFOIL_NAME = "naca2412"
OPT_ALPHA = 4.0
OPT_RE = 2.0e6

all_airfoils = TRAIN_AIRFOILS + VAL_AIRFOILS + TEST_AIRFOILS
target = [a for a in all_airfoils if a["name"] == TARGET_AIRFOIL_NAME][0]

case_dir = Path(OPT_OUT_DIR) / f"{TARGET_AIRFOIL_NAME}_a{int(OPT_ALPHA)}_Re{int(OPT_RE)}"
xai_dir = case_dir / "xai"
xai_dir.mkdir(parents=True, exist_ok=True)

if "best_params" not in globals():
    opt_json = case_dir / "optimized_parsec.json"
    data = json.loads(opt_json.read_text())
    best_params = data["optimized_parsec"]

xai_report = run_xai_suite(
    model=model,
    geometry=target["geometry"],
    parsec_params=best_params,
    alpha_deg=OPT_ALPHA,
    Re_phys=OPT_RE,
    xlim=XLIM,
    ylim=YLIM,
    loss_weights=adaptive_weights.all_weights() if "adaptive_weights" in globals() and adaptive_weights is not None else None,
    out_dir=xai_dir,
    device=device,
)